# 🤖 Préprocessing et Modélisation Professionnelle

## 🎯 Objectifs d'apprentissage

À la fin de ce notebook, vous maîtriserez :
- 🧹 **Preprocessing rigoureux** : Imputation, encodage, standardisation
- ⚠️ **Prévention du data leakage** : fit_transform sur train, transform sur test
- 📂 **Séparation stratifiée** : Maintien des distributions
- 🤖 **Modélisation complète** : Baseline → Optimisation → Validation
- 📊 **Évaluation multicritères** : Accuracy, Precision, Recall, F1, AUC
- 💾 **Sauvegarde professionnelle** : Modèles, métadonnées, traçabilité

## 🔄 Pipeline ML - Étapes 2 à 6

1. Explorer ✅
2. **Préparer** ← *Nous commençons ici*
3. **Séparer**
4. **Choisir & Entraîner**
5. **Optimiser**
6. **Évaluer**

## ⚠️ Règle d'or
**Standardiser sur le train uniquement (fit_transform), puis appliquer sur le test (transform)**

Ceci évite la **fuite de données (data leakage)** !

In [1]:
# 📚 Importation des bibliothèques principales

# Manipulation et visualisation des données
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Scikit-learn
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, 
                           roc_auc_score, mean_squared_error, r2_score, silhouette_score,
                           precision_score, recall_score, f1_score, roc_curve)

# Utilitaires
import warnings
import joblib
import json
from datetime import datetime

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ Bibliothèques importées avec succès !")
print("\n📊 Manipulation : pandas, numpy")
print("🎨 Visualisation : matplotlib, seaborn")
print("🤖 Machine Learning : scikit-learn (complet)")
print("\n🔧 Composants disponibles :")
print("  • Preprocessing : StandardScaler, LabelEncoder, PolynomialFeatures")
print("  • Model Selection : train_test_split, GridSearchCV, RandomizedSearchCV")
print("  • Models : Logistic, Ridge, Lasso, ElasticNet, RF, GBM, SVM, KNN")
print("  • Clustering : KMeans, DBSCAN, Agglomerative")
print("  • Metrics : Accuracy, Precision, Recall, AUC, MSE, R2, Silhouette")

✅ Bibliothèques importées avec succès !

📊 Manipulation : pandas, numpy
🎨 Visualisation : matplotlib, seaborn
🤖 Machine Learning : scikit-learn (complet)

🔧 Composants disponibles :
  • Preprocessing : StandardScaler, LabelEncoder, PolynomialFeatures
  • Model Selection : train_test_split, GridSearchCV, RandomizedSearchCV
  • Models : Logistic, Ridge, Lasso, ElasticNet, RF, GBM, SVM, KNN
  • Clustering : KMeans, DBSCAN, Agglomerative
  • Metrics : Accuracy, Precision, Recall, AUC, MSE, R2, Silhouette


## 📂 Chargement et Préparation Initiale

### Objectifs :
- Charger les données brutes
- Séparer features et target
- Identifier les types de variables

In [2]:
# Chargement des données
print("📂 CHARGEMENT DES DONNÉES")
print("="*50)

df = pd.read_csv('../data/heart_disease_dataset.csv')

# Informations générales
print(f"📊 Dataset : {df.shape[0]} patients × {df.shape[1]} caractéristiques")
print(f"📝 Colonnes : {list(df.columns)}")

# Séparation features et target
target_col = 'Heart Disease'
X = df.drop(target_col, axis=1)
y = df[target_col]

print(f"\n🎯 Séparation Features/Target :")
print(f"  • Features (X) : {X.shape}")
print(f"  • Target (y) : {y.shape}")
print(f"  • Target : {target_col}")

# Identification des types de variables
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"\n🔍 Types de variables :")
print(f"  • Numériques ({len(numeric_features)}) : {numeric_features}")
print(f"  • Catégorielles ({len(categorical_features)}) : {categorical_features}")

# Distribution du target
print(f"\n📊 Distribution du target :")
target_distribution = y.value_counts(normalize=True).round(3)
for val, pct in target_distribution.items():
    print(f"  • Classe {val} : {pct*100:.1f}%")

📂 CHARGEMENT DES DONNÉES
📊 Dataset : 1000 patients × 16 caractéristiques
📝 Colonnes : ['Age', 'Gender', 'Cholesterol', 'Blood Pressure', 'Heart Rate', 'Smoking', 'Alcohol Intake', 'Exercise Hours', 'Family History', 'Diabetes', 'Obesity', 'Stress Level', 'Blood Sugar', 'Exercise Induced Angina', 'Chest Pain Type', 'Heart Disease']

🎯 Séparation Features/Target :
  • Features (X) : (1000, 15)
  • Target (y) : (1000,)
  • Target : Heart Disease

🔍 Types de variables :
  • Numériques (7) : ['Age', 'Cholesterol', 'Blood Pressure', 'Heart Rate', 'Exercise Hours', 'Stress Level', 'Blood Sugar']
  • Catégorielles (8) : ['Gender', 'Smoking', 'Alcohol Intake', 'Family History', 'Diabetes', 'Obesity', 'Exercise Induced Angina', 'Chest Pain Type']

📊 Distribution du target :
  • Classe 0 : 60.8%
  • Classe 1 : 39.2%


## 📂 Séparation Train/Test avec Stratification

### Objectifs :
- Séparer les données en ensembles d'entraînement et de test
- Maintenir la distribution du target (stratification)
- Isoler le test set pour évaluation finale uniquement

### ⚠️ Règle importante
**Le test set ne doit JAMAIS être utilisé avant l'évaluation finale !**

In [ ]:
# Séparation stratifiée train/test
print("📂 SÉPARATION STRATIFIÉE TRAIN/TEST")
print("="*50)

# Séparation avec stratification pour préserver la distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,           # 20% pour le test
    random_state=42,         # Reproductibilité
    stratify=y               # Stratification importante !
)

print(f"📊 Répartition des données :")
print(f"  • Train set : {X_train.shape[0]} patients ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"  • Test set : {X_test.shape[0]} patients ({X_test.shape[0]/len(df)*100:.1f}%)")

# Vérification de la stratification
print(f"\n🎯 Vérification de la stratification :")
train_dist = y_train.value_counts(normalize=True).round(3)
test_dist = y_test.value_counts(normalize=True).round(3)

print(f"\n📊 Distribution target - Train :")
for val, pct in train_dist.items():
    print(f"  • Classe {val} : {pct*100:.1f}%")

print(f"\n📊 Distribution target - Test :")
for val, pct in test_dist.items():
    print(f"  • Classe {val} : {pct*100:.1f}%")

# Calcul de la différence de distribution
diff = abs(train_dist - test_dist).max()
if diff < 0.05:
    print(f"\n✅ Stratification réussie ! Différence max : {diff:.3f}")
else:
    print(f"\n⚠️ Attention : Différence de distribution : {diff:.3f}")

print(f"\n🔒 TEST SET ISOLÉ : Ne sera utilisé que pour l'évaluation finale !")

## 🧹 Préprocessing des Données

### Objectifs :
- Imputer les valeurs manquantes
- Encoder les variables catégorielles
- Standardiser les variables numériques
- **Prévenir le data leakage**

In [ ]:
# Vérification des valeurs manquantes avant preprocessing
print("🔍 VALEURS MANQUANTES - AVANT PREPROCESSING")
print("="*50)

missing_train = X_train.isnull().sum()
missing_test = X_test.isnull().sum()

print(f"📊 Valeurs manquantes - Train :")
missing_train_cols = missing_train[missing_train > 0]
if len(missing_train_cols) > 0:
    print(missing_train_cols)
else:
    print("  ✅ Aucune valeur manquante")

print(f"\n📊 Valeurs manquantes - Test :")
missing_test_cols = missing_test[missing_test > 0]
if len(missing_test_cols) > 0:
    print(missing_test_cols)
else:
    print("  ✅ Aucune valeur manquante")

# Stratégie d'imputation
print(f"\n💡 STRATÉGIE D'IMPUTATION :")
print(f"  • Variables numériques : Médiane (robuste aux outliers)")
print(f"  • Variables catégorielles : Mode (valeur la plus fréquente)")

In [ ]:
# Imputation des valeurs manquantes
print("🔧 IMPUTATION DES VALEURS MANQUANTES")
print("="*50)

# Copie pour éviter de modifier les originaux
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

# Dictionnaire pour stocker les valeurs d'imputation
imputation_values = {}

# Imputation des variables numériques
if numeric_features:
    print(f"\n🔢 Imputation variables numériques :")
    for col in numeric_features:
        if X_train_imputed[col].isnull().sum() > 0:
            # Calcul de la médiane sur le TRAIN SEULEMENT
            median_val = X_train_imputed[col].median()
            imputation_values[col] = median_val
            
            # Imputation sur train et test
            X_train_imputed[col].fillna(median_val, inplace=True)
            X_test_imputed[col].fillna(median_val, inplace=True)
            
            print(f"  • {col}: médiane = {median_val:.3f}")

# Imputation des variables catégorielles
if categorical_features:
    print(f"\n📝 Imputation variables catégorielles :")
    for col in categorical_features:
        if X_train_imputed[col].isnull().sum() > 0:
            # Calcul du mode sur le TRAIN SEULEMENT
            mode_val = X_train_imputed[col].mode()[0]
            imputation_values[col] = mode_val
            
            # Imputation sur train et test
            X_train_imputed[col].fillna(mode_val, inplace=True)
            X_test_imputed[col].fillna(mode_val, inplace=True)
            
            print(f"  • {col}: mode = {mode_val}")

# Vérification finale
print(f"\n✅ Vérification post-imputation :")
print(f"  • Train : {X_train_imputed.isnull().sum().sum()} valeurs manquantes")
print(f"  • Test : {X_test_imputed.isnull().sum().sum()} valeurs manquantes")

if X_train_imputed.isnull().sum().sum() == 0 and X_test_imputed.isnull().sum().sum() == 0:
    print(f"\n🎉 Imputation réussie ! Aucune valeur manquante restante.")

In [ ]:
# Encodage des variables catégorielles
print("🔄 ENCODAGE DES VARIABLES CATÉGORIELLES")
print("="*50)

# Dictionnaire pour stocker les encodeurs
label_encoders = {}
X_train_encoded = X_train_imputed.copy()
X_test_encoded = X_test_imputed.copy()

if categorical_features:
    print(f"\n📝 Encodage des {len(categorical_features)} variables catégorielles :")
    
    for col in categorical_features:
        print(f"\n🔧 Encodage de '{col}' :")
        
        # Création et entraînement du label encoder
        le = LabelEncoder()
        
        # FIT et TRANSFORM sur le TRAIN
        X_train_encoded[col] = le.fit_transform(X_train_encoded[col])
        
        # TRANSFORM SEULEMENT sur le TEST
        X_test_encoded[col] = le.transform(X_test_encoded[col])
        
        # Sauvegarde de l'encodeur
        label_encoders[col] = le
        
        # Affichage des informations
        print(f"  • Classes uniques : {le.classes_}")
        print(f"  • Mapping : {dict(zip(le.classes_, le.transform(le.classes_)))}")
        print(f"  • Valeurs encodées : {sorted(X_train_encoded[col].unique())}")
else:
    print("📝 Aucune variable catégorielle à encoder")

# Vérification finale
print(f"\n✅ Types de données après encodage :")
print(X_train_encoded.dtypes.value_counts())
print(f"\n📊 Dimensions finales :")
print(f"  • Train : {X_train_encoded.shape}")
print(f"  • Test : {X_test_encoded.shape}")

In [ ]:
# Standardisation des variables numériques (RÈGLE D'OR !)
print("⚖️ STANDARDISATION (PRÉVENTION DU DATA LEAKAGE)")
print("="*60)

print("⚠️ RÈGLE IMPORTANTE :")
print("  • fit_transform() sur le TRAIN set")
print("  • transform() SEULEMENT sur le TEST set")
print("  • Ceci évite la fuite de données du test vers le train !\n")

# Création du scaler
scaler = StandardScaler()

# Standardisation
if numeric_features:
    print(f"\n🔢 Standardisation des {len(numeric_features)} variables numériques :")
    
    # FIT et TRANSFORM sur le TRAIN
    X_train_scaled_array = scaler.fit_transform(X_train_encoded[numeric_features])
    print(f"  ✅ fit_transform appliqué sur train")
    
    # TRANSFORM SEULEMENT sur le TEST
    X_test_scaled_array = scaler.transform(X_test_encoded[numeric_features])
    print(f"  ✅ transform appliqué sur test")
    
    # Reconversion en DataFrame
    X_train_scaled = pd.DataFrame(X_train_scaled_array, columns=numeric_features, index=X_train_encoded.index)
    X_test_scaled = pd.DataFrame(X_test_scaled_array, columns=numeric_features, index=X_test_encoded.index)
    
    # Combinaison avec variables catégorielles
    if categorical_features:
        X_train_final = pd.concat([X_train_scaled, X_train_encoded[categorical_features]], axis=1)
        X_test_final = pd.concat([X_test_scaled, X_test_encoded[categorical_features]], axis=1)
    else:
        X_train_final = X_train_scaled
        X_test_final = X_test_scaled
    
    # Vérification de la standardisation
    print(f"\n📊 Vérification de la standardisation (train) :")
    print(f"  • Moyennes : {np.round(X_train_scaled.mean(), 3)}")
    print(f"  • Écarts-types : {np.round(X_train_scaled.std(), 3)}")
    
    print(f"\n📊 Vérification de la standardisation (test) :")
    print(f"  • Moyennes : {np.round(X_test_scaled.mean(), 3)}")
    print(f"  • Écarts-types : {np.round(X_test_scaled.std(), 3)}")
    
else:
    X_train_final = X_train_encoded
    X_test_final = X_test_encoded
    print("📝 Aucune variable numérique à standardiser")

print(f"\n🎯 Dimensions finales des données prétraitées :")
print(f"  • Train : {X_train_final.shape}")
print(f"  • Test : {X_test_final.shape}")
print(f"\n✅ Preprocessing complété avec succès !")
print(f"🔒 Data leakage prévenu avec succès !")

## 🤖 Modélisation - Baseline

### Objectifs :
- Établir des modèles de référence simples
- Comparer différentes approches algorithmiques
- Identifier les meilleures performances initiales
- Détecter les surapprentissages potentiels

In [ ]:
# Définition des modèles de baseline
print("🤖 DÉFINITION DES MODÈLES DE BASELINE")
print("="*50)

# Dictionnaire des modèles
baseline_models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100),
    'SVM': SVC(random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

print("📋 Modèles de baseline définis :")
for name, model in baseline_models.items():
    print(f"  🤖 {name} : {model.__class__.__name__}")

# Dictionnaire pour stocker les résultats
baseline_results = {}
print(f"\n✅ {len(baseline_models)} modèles prêts pour l'entraînement !")

In [ ]:
# Entraînement et évaluation des modèles de baseline
print("🚀 ENTRAÎNEMENT DES MODÈLES DE BASELINE")
print("="*50)

for name, model in baseline_models.items():
    print(f"\n🤖 Entraînement de {name}...")
    
    # Entraînement
    model.fit(X_train_final, y_train)
    
    # Prédictions
    y_train_pred = model.predict(X_train_final)
    y_test_pred = model.predict(X_test_final)
    
    # Probabilités pour AUC
    if hasattr(model, 'predict_proba'):
        y_train_proba = model.predict_proba(X_train_final)[:, 1]
        y_test_proba = model.predict_proba(X_test_final)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_train_proba = model.decision_function(X_train_final)
        y_test_proba = model.decision_function(X_test_final)
    else:
        y_train_proba = None
        y_test_proba = None
    
    # Calcul des métriques
    metrics = {
        'model': model,
        'train_accuracy': accuracy_score(y_train, y_train_pred),
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'train_precision': precision_score(y_train, y_train_pred, average='weighted'),
        'test_precision': precision_score(y_test, y_test_pred, average='weighted'),
        'train_recall': recall_score(y_train, y_train_pred, average='weighted'),
        'test_recall': recall_score(y_test, y_test_pred, average='weighted'),
        'train_f1': f1_score(y_train, y_train_pred, average='weighted'),
        'test_f1': f1_score(y_test, y_test_pred, average='weighted')
    }
    
    # AUC si disponible
    if y_test_proba is not None:
        metrics['train_auc'] = roc_auc_score(y_train, y_train_proba)
        metrics['test_auc'] = roc_auc_score(y_test, y_test_proba)
    else:
        metrics['train_auc'] = None
        metrics['test_auc'] = None
    
    baseline_results[name] = metrics
    
    # Affichage des résultats
    print(f"  ✅ Train Accuracy : {metrics['train_accuracy']:.4f}")
    print(f"  ✅ Test Accuracy : {metrics['test_accuracy']:.4f}")
    print(f"  📊 Test Precision : {metrics['test_precision']:.4f}")
    print(f"  📊 Test Recall : {metrics['test_recall']:.4f}")
    print(f"  📊 Test F1 : {metrics['test_f1']:.4f}")
    if metrics['test_auc'] is not None:
        print(f"  📈 Test AUC : {metrics['test_auc']:.4f}")
    
    # Détection du surapprentissage
    overfitting = metrics['train_accuracy'] - metrics['test_accuracy']
    if overfitting > 0.1:
        print(f"  ⚠️ Surapprentissage détecté : {overfitting:.4f}")
    elif overfitting > 0.05:
        print(f"  ⚠️ Léger surapprentissage : {overfitting:.4f}")
    else:
        print(f"  ✅ Bon équilibre train/test")

print(f"\n🎉 Entraînement baseline complété !")
print(f"📊 {len(baseline_results)} modèles évalués")

In [ ]:
# Comparaison visuelle des modèles baseline
print("📊 COMPARAISON VISUELLE DES MODÈLES BASELINE")
print("="*50)

# Création d'un DataFrame de comparaison
comparison_data = []
for name, metrics in baseline_results.items():
    comparison_data.append({
        'Modèle': name,
        'Accuracy Test': metrics['test_accuracy'],
        'Precision Test': metrics['test_precision'],
        'Recall Test': metrics['test_recall'],
        'F1 Test': metrics['test_f1'],
        'AUC Test': metrics['test_auc'] if metrics['test_auc'] else 0
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Accuracy Test', ascending=False)

# Visualisation
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('📊 Comparaison des Modèles Baseline', fontsize=16, fontweight='bold')

# Accuracy
axes[0, 0].bar(comparison_df['Modèle'], comparison_df['Accuracy Test'], color='skyblue')
axes[0, 0].set_title('Accuracy Test')
axes[0, 0].set_ylabel('Score')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].set_ylim(0, 1)

# Precision
axes[0, 1].bar(comparison_df['Modèle'], comparison_df['Precision Test'], color='lightgreen')
axes[0, 1].set_title('Precision Test')
axes[0, 1].set_ylabel('Score')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].set_ylim(0, 1)

# Recall
axes[0, 2].bar(comparison_df['Modèle'], comparison_df['Recall Test'], color='orange')
axes[0, 2].set_title('Recall Test')
axes[0, 2].set_ylabel('Score')
axes[0, 2].tick_params(axis='x', rotation=45)
axes[0, 2].set_ylim(0, 1)

# F1
axes[1, 0].bar(comparison_df['Modèle'], comparison_df['F1 Test'], color='red')
axes[1, 0].set_title('F1 Test')
axes[1, 0].set_ylabel('Score')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].set_ylim(0, 1)

# AUC
axes[1, 1].bar(comparison_df['Modèle'], comparison_df['AUC Test'], color='purple')
axes[1, 1].set_title('AUC Test')
axes[1, 1].set_ylabel('Score')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].set_ylim(0, 1)

# Tableau récapitulatif
axes[1, 2].axis('off')
table_data = comparison_df.round(3).values
table = axes[1, 2].table(cellText=table_data, colLabels=comparison_df.columns,
                          rowLabels=comparison_df['Modèle'], cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 2)
axes[1, 2].set_title('Tableau Récapitulatif', fontsize=12, pad=20)

plt.tight_layout()
plt.show()

# Affichage du classement
print("\n🏆 CLASSEMENT DES MODÈLES BASELINE :")
print(comparison_df.to_string(index=False, float_format='%.3f'))

# Meilleur modèle baseline
best_baseline = comparison_df.iloc[0]
print(f"\n🥇 MEILLEUR MODÈLE BASELINE : {best_baseline['Modèle']}")
print(f"🎯 Accuracy : {best_baseline['Accuracy Test']:.4f}")

## 🔧 Optimisation des Hyperparamètres

### Objectifs :
- Améliorer les performances des meilleurs modèles
- Utiliser GridSearchCV pour optimisation systématique
- Validation croisée pour éviter le surapprentissage
- Comparer baseline vs optimisé

In [ ]:
# Sélection des meilleurs modèles pour optimisation
print("🎯 SÉLECTION DES MODÈLES À OPTIMISER")
print("="*50)

# Prendre les 3 meilleurs modèles selon accuracy
top_models = comparison_df.head(3)['Modèle'].tolist()
print(f"🏆 Modèles sélectionnés pour optimisation : {top_models}")

# Définition des grilles de paramètres
param_grids = {
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'Gradient Boosting': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'subsample': [0.8, 0.9, 1.0]
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 'auto'],
        'kernel': ['rbf', 'linear']
    }
}

print("\n🔧 Grilles de paramètres définies :")
for model_name, params in param_grids.items():
    if model_name in top_models:
        print(f"\n🤖 {model_name} :")
        total_combinations = 1
        for param, values in params.items():
            total_combinations *= len(values)
            print(f"  • {param}: {values}")
        print(f"  📊 Combinaisons totales : {total_combinations}")

In [ ]:
# Optimisation avec GridSearchCV
print("🔍 OPTIMISATION AVEC GRIDSEARCHCV")
print("="*50)

# Configuration de la validation croisée
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"📊 Validation croisée : {cv.get_n_splits()}-fold stratifiée")

# Dictionnaire pour stocker les résultats optimisés
optimized_results = {}

for model_name in top_models:
    if model_name in param_grids:
        print(f"\n🔍 Optimisation de {model_name}...")
        
        # Récupérer le modèle de base
        base_model = baseline_results[model_name]['model']
        
        # Configuration de GridSearchCV
        grid_search = GridSearchCV(
            estimator=base_model,
            param_grid=param_grids[model_name],
            cv=cv,
            scoring='accuracy',
            n_jobs=-1,  # Utiliser tous les processeurs
            verbose=1
        )
        
        # Entraînement avec validation croisée
        print(f"  🔄 Recherche des meilleurs paramètres...")
        grid_search.fit(X_train_final, y_train)
        
        # Meilleurs résultats
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
        best_score = grid_search.best_score_
        
        # Évaluation sur le test set
        y_test_pred_opt = best_model.predict(X_test_final)
        test_accuracy_opt = accuracy_score(y_test, y_test_pred_opt)
        
        # Calcul de l'amélioration
        baseline_accuracy = baseline_results[model_name]['test_accuracy']
        improvement = test_accuracy_opt - baseline_accuracy
        
        # Stockage des résultats
        optimized_results[model_name] = {
            'model': best_model,
            'best_params': best_params,
            'cv_score': best_score,
            'test_accuracy': test_accuracy_opt,
            'baseline_accuracy': baseline_accuracy,
            'improvement': improvement
        }
        
        # Affichage des résultats
        print(f"  ✅ Meilleurs paramètres : {best_params}")
        print(f"  📊 Score CV moyen : {best_score:.4f}")
        print(f"  🎯 Accuracy Test : {test_accuracy_opt:.4f}")
        print(f"  📈 Amélioration : {improvement:+.4f} ({improvement/baseline_accuracy*100:+.1f}%)")

print(f"\n🎉 Optimisation terminée !")
print(f"📊 {len(optimized_results)} modèles optimisés")

## 📈 Évaluation Finale et Comparaison

### Objectifs :
- Comparer baseline vs optimisé
- Analyse détaillée du meilleur modèle
- Visualisations des performances
- Interprétation des résultats

In [ ]:
# Comparaison complète baseline vs optimisé
print("📊 COMPARAISON COMPLÈTE BASELINE VS OPTIMISÉ")
print("="*50)

# Création du tableau de comparaison final
final_comparison = []

# Ajouter les modèles baseline
for name, metrics in baseline_results.items():
    final_comparison.append({
        'Modèle': f"{name} (Baseline)",
        'Type': 'Baseline',
        'Accuracy': metrics['test_accuracy'],
        'Precision': metrics['test_precision'],
        'Recall': metrics['test_recall'],
        'F1': metrics['test_f1'],
        'AUC': metrics['test_auc'] if metrics['test_auc'] else 0
    })

# Ajouter les modèles optimisés
for name, opt_metrics in optimized_results.items():
    opt_model = opt_metrics['model']
    opt_pred = opt_model.predict(X_test_final)
    
    # Calcul de toutes les métriques
    final_comparison.append({
        'Modèle': f"{name} (Optimisé)",
        'Type': 'Optimisé',
        'Accuracy': opt_metrics['test_accuracy'],
        'Precision': precision_score(y_test, opt_pred, average='weighted'),
        'Recall': recall_score(y_test, opt_pred, average='weighted'),
        'F1': f1_score(y_test, opt_pred, average='weighted'),
        'AUC': roc_auc_score(y_test, opt_model.predict_proba(X_test_final)[:, 1]) if hasattr(opt_model, 'predict_proba') else 0
    })

# Création du DataFrame final
final_df = pd.DataFrame(final_comparison)
final_df = final_df.sort_values('Accuracy', ascending=False)

# Affichage du classement final
print("\n🏆 CLASSEMENT FINAL DES MODÈLES :")
print(final_df.to_string(index=False, float_format='%.4f'))

# Meilleur modèle final
best_final = final_df.iloc[0]
best_model_name = best_final['Modèle']
best_accuracy = best_final['Accuracy']

print(f"\n🥇 MEILLEUR MODÈLE FINAL : {best_model_name}")
print(f"🎯 Accuracy : {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print(f"📊 Type : {best_final['Type']}")
print(f"📈 AUC : {best_final['AUC']:.4f}")

In [ ]:
# Analyse détaillée du meilleur modèle
print(f"🔍 ANALYSE DÉTAILLÉE : {best_model_name}")
print("="*60)

# Récupérer le meilleur modèle
if '(Optimisé)' in best_model_name:
    base_name = best_model_name.replace(' (Optimisé)', '')
    best_model = optimized_results[base_name]['model']
else:
    base_name = best_model_name.replace(' (Baseline)', '')
    best_model = baseline_results[base_name]['model']

# Prédictions finales
y_pred_final = best_model.predict(X_test_final)
y_proba_final = None

if hasattr(best_model, 'predict_proba'):
    y_proba_final = best_model.predict_proba(X_test_final)[:, 1]
elif hasattr(best_model, 'decision_function'):
    y_proba_final = best_model.decision_function(X_test_final)

# Métriques complètes
print("\n📊 MÉTRIQUES COMPLÈTES :")
print(f"  Accuracy : {accuracy_score(y_test, y_pred_final):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred_final):.4f}")
print(f"  Recall : {recall_score(y_test, y_pred_final):.4f}")
print(f"  F1-Score : {f1_score(y_test, y_pred_final):.4f}")
if y_proba_final is not None:
    print(f"  AUC : {roc_auc_score(y_test, y_proba_final):.4f}")

# Rapport de classification détaillé
print("\n📋 RAPPORT DE CLASSIFICATION :")
print(classification_report(y_test, y_pred_final, target_names=['Non Malade', 'Malade']))

# Paramètres optimaux si modèle optimisé
if '(Optimisé)' in best_model_name and base_name in optimized_results:
    print(f"\n🔧 PARAMÈTRES OPTIMAUX :")
    for param, value in optimized_results[base_name]['best_params'].items():
        print(f"  • {param}: {value}")

In [ ]:
# Visualisations du meilleur modèle
print(f"📊 VISUALISATIONS : {best_model_name}")
print("="*50)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(f'🏆 Analyse du Meilleur Modèle : {best_model_name}', fontsize=16, fontweight='bold')

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Prédit Non', 'Prédit Oui'],
            yticklabels=['Réel Non', 'Réel Oui'], ax=axes[0, 0])
axes[0, 0].set_title('Matrice de Confusion')
axes[0, 0].set_xlabel('Prédiction')
axes[0, 0].set_ylabel('Réalité')

# Courbe ROC
if y_proba_final is not None:
    fpr, tpr, _ = roc_curve(y_test, y_proba_final)
    auc_score = roc_auc_score(y_test, y_proba_final)
    axes[0, 1].plot(fpr, tpr, color='red', lw=2, label=f'ROC (AUC = {auc_score:.3f})')
    axes[0, 1].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[0, 1].set_xlim([0.0, 1.0])
    axes[0, 1].set_ylim([0.0, 1.05])
    axes[0, 1].set_xlabel('Taux de Faux Positifs')
    axes[0, 1].set_ylabel('Taux de Vrais Positifs')
    axes[0, 1].set_title('Courbe ROC')
    axes[0, 1].legend(loc="lower right")
else:
    axes[0, 1].text(0.5, 0.5, 'Courbe ROC\nnon disponible', 
                    ha='center', va='center', transform=axes[0, 1].transAxes)
    axes[0, 1].set_title('Courbe ROC')

# Importance des features (si disponible)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X_train_final.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    # Top 10 features
    top_features = feature_importance.head(10)
    
    axes[1, 0].barh(range(len(top_features)), top_features['Importance'])
    axes[1, 0].set_yticks(range(len(top_features)))
    axes[1, 0].set_yticklabels(top_features['Feature'])
    axes[1, 0].set_xlabel('Importance')
    axes[1, 0].set_title('Top 10 Features')
    axes[1, 0].invert_yaxis()
else:
    axes[1, 0].text(0.5, 0.5, 'Importance des features\nnon disponible', 
                    ha='center', va='center', transform=axes[1, 0].transAxes)
    axes[1, 0].set_title('Importance des Features')

# Comparaison des métriques
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1']
baseline_metrics = []
optimized_metrics = []

# Trouver les métriques correspondantes
base_name = best_model_name.replace(' (Optimisé)', '').replace(' (Baseline)', '')
if base_name in baseline_results:
    baseline_metrics = [
        baseline_results[base_name]['test_accuracy'],
        baseline_results[base_name]['test_precision'],
        baseline_results[base_name]['test_recall'],
        baseline_results[base_name]['test_f1']
    ]

if base_name in optimized_results:
    opt_model = optimized_results[base_name]['model']
    opt_pred = opt_model.predict(X_test_final)
    optimized_metrics = [
        accuracy_score(y_test, opt_pred),
        precision_score(y_test, opt_pred, average='weighted'),
        recall_score(y_test, opt_pred, average='weighted'),
        f1_score(y_test, opt_pred, average='weighted')
    ]

x = np.arange(len(metrics_names))
width = 0.35

if baseline_metrics:
    axes[1, 1].bar(x - width/2, baseline_metrics, width, label='Baseline', color='lightblue')
if optimized_metrics:
    axes[1, 1].bar(x + width/2, optimized_metrics, width, label='Optimisé', color='lightcoral')

axes[1, 1].set_xlabel('Métrique')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_title('Baseline vs Optimisé')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(metrics_names)
axes[1, 1].legend()
axes[1, 1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

# Afficher les features les plus importantes
if hasattr(best_model, 'feature_importances_'):
    print("\n🎯 TOP 10 FEATURES LES PLUS IMPORTANTES :")
    print(feature_importance.head(10).to_string(index=False))

## 💾 Sauvegarde Professionnelle

### Objectifs :
- Sauvegarder le meilleur modèle et composants
- Créer des métadonnées complètes
- Assurer la traçabilité et reproductibilité
- Préparer pour déploiement

In [ ]:
# Création du dossier models si nécessaire
import os
if not os.path.exists('../models'):
    os.makedirs('../models')
    print("📁 Dossier 'models' créé")

# Sauvegarde du meilleur modèle
print("💾 SAUVEGARDE DU MODÈLE FINAL")
print("="*50)

# Sauvegarder le meilleur modèle
model_path = '../models/best_heart_model_professional.pkl'
joblib.dump(best_model, model_path)
print(f"💾 Meilleur modèle sauvegardé : {model_path}")

# Sauvegarder le scaler
scaler_path = '../models/scaler_professional.pkl'
joblib.dump(scaler, scaler_path)
print(f"💾 Scaler sauvegardé : {scaler_path}")

# Sauvegarder les label encoders
encoders_path = '../models/label_encoders_professional.pkl'
joblib.dump(label_encoders, encoders_path)
print(f"💾 Label encoders sauvegardés : {encoders_path}")

# Sauvegarder les valeurs d'imputation
imputation_path = '../models/imputation_values_professional.pkl'
joblib.dump(imputation_values, imputation_path)
print(f"💾 Valeurs d'imputation sauvegardées : {imputation_path}")

In [ ]:
# Création des métadonnées complètes
print("📋 CRÉATION DES MÉTADONNÉES")
print("="*50)

# Métadonnées complètes
metadata = {
    # Informations projet
    'project_info': {
        'name': 'Heart Disease Prediction - Professional Pipeline',
        'version': '2.0',
        'created_date': datetime.now().isoformat(),
        'author': 'Professional ML Pipeline',
        'domain': 'Medical / Cardiology'
    },
    
    # Informations dataset
    'dataset_info': {
        'total_samples': len(df),
        'total_features': df.shape[1] - 1,  # -1 pour le target
        'target_column': target_col,
        'numeric_features': numeric_features,
        'categorical_features': categorical_features,
        'missing_values_before': df.isnull().sum().sum(),
        'missing_values_after': 0
    },
    
    # Informations preprocessing
    'preprocessing': {
        'train_size': len(X_train),
        'test_size': len(X_test),
        'test_size_ratio': len(X_test) / len(df),
        'stratification': True,
        'imputation_strategy': {
            'numeric': 'median',
            'categorical': 'mode'
        },
        'encoding': 'LabelEncoder',
        'scaling': 'StandardScaler',
        'data_leakage_prevention': True
    },
    
    # Informations modèle
    'model_info': {
        'best_model_name': best_model_name,
        'model_type': type(best_model).__name__,
        'final_accuracy': float(best_accuracy),
        'final_precision': float(precision_score(y_test, y_pred_final)),
        'final_recall': float(recall_score(y_test, y_pred_final)),
        'final_f1': float(f1_score(y_test, y_pred_final)),
        'final_auc': float(roc_auc_score(y_test, y_proba_final)) if y_proba_final is not None else None
    },
    
    # Résultats comparatifs
    'comparison_results': {
        'baseline_models': len(baseline_results),
        'optimized_models': len(optimized_results),
        'improvement_achieved': '(Optimisé)' in best_model_name
    },
    
    # Paramètres optimaux (si applicable)
    'optimal_parameters': optimized_results[base_name]['best_params'] if '(Optimisé)' in best_model_name and base_name in optimized_results else None,
    
    # Importance des features (si disponible)
    'feature_importance': None
}

# Ajouter l'importance des features si disponible
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train_final.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False).to_dict('records')
    metadata['feature_importance'] = feature_importance

# Sauvegarder les métadonnées
with open('../models/model_metadata_professional.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"💾 Métadonnées sauvegardées : ../models/model_metadata_professional.json")
print(f"\n📊 Résumé des métadonnées :")
print(f"  • Projet : {metadata['project_info']['name']}")
print(f"  • Dataset : {metadata['dataset_info']['total_samples']} échantillons")
print(f"  • Meilleur modèle : {metadata['model_info']['best_model_name']}")
print(f"  • Accuracy finale : {metadata['model_info']['final_accuracy']:.4f}")
print(f"  • Data leakage prévenu : {metadata['preprocessing']['data_leakage_prevention']}")

## 🎯 Synthèse et Recommandations Finale

### Résumé du pipeline ML professionnel
- ✅ **Exploration** : Analyse complète des données médicales
- ✅ **Preprocessing** : Imputation, encodage, standardisation avec prévention du data leakage
- ✅ **Modélisation** : 5 modèles baseline + optimisation des 3 meilleurs
- ✅ **Évaluation** : Métriques multiples adaptées au contexte médical
- ✅ **Sauvegarde** : Modèles, composants et métadonnées complètes

### Recommandations pour la suite
1. **Validation externe** : Tester sur dataset indépendant
2. **Monitoring** : Surveiller les performances en production
3. **Mise à jour** : Réentraînement périodique avec nouvelles données
4. **Interprétation** : Analyse SHAP pour explications locales
5. **Déploiement** : API REST pour intégration clinique

In [ ]:
# Synthèse finale du projet
print("🎯 SYNTHÈSE FINALE DU PIPELINE ML PROFESSIONNEL")
print("="*60)

print("\n📊 RÉSULTATS PRINCIPAUX :")
print(f"  • Dataset analysé : {len(df):,} patients")
print(f"  • Features utilisées : {X_train_final.shape[1]}")
print(f"  • Modèles évalués : {len(baseline_results)} baseline + {len(optimized_results)} optimisés")
print(f"  • Meilleur modèle : {best_model_name}")
print(f"  • Performance finale : {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")

print("\n🔧 PROCESSUS ML RESPECTÉ :")
print(f"  ✅ Exploration des données complétée")
print(f"  ✅ Preprocessing rigoureux avec prévention du data leakage")
print(f"  ✅ Séparation stratifiée train/test")
print(f"  ✅ Modélisation baseline → optimisation")
print(f"  ✅ Évaluation multicritères (Accuracy, Precision, Recall, F1, AUC)")
print(f"  ✅ Sauvegarde professionnelle avec métadonnées")

print("\n🏥 CONTEXTE MÉDICAL :")
print(f"  • Domaine : Cardiologie prédictive")
print(f"  • Objectif : Aide au dépistage précoce")
print(f"  • Impact : Potentiellement sauver des vies")
print(f"  • Recommandation : Validation clinique requise avant déploiement")

print("\n📈 COMPÉTENCES ACQUISES :")
print(f"  ✅ Pipeline ML complet et reproductible")
print(f"  ✅ Prévention du data leakage")
print(f"  ✅ Optimisation systématique des hyperparamètres")
print(f"  ✅ Évaluation multicritères contextualisée")
print(f"  ✅ Sauvegarde professionnelle et traçabilité")
print(f"  ✅ Interprétation métier des résultats")

print("\n🚀 PROCHAINES ÉTAPES :")
print(f"  1. Validation sur dataset externe indépendant")
print(f"  2. Analyse SHAP pour interprétabilité locale")
print(f"  3. Développement API REST pour déploiement")
print(f"  4. Monitoring et maintenance en production")
print(f"  5. Validation clinique et réglementaire")

print("\n🌟 FÉLICITATIONS !")
print(f"Vous avez maîtrisé un pipeline ML professionnel complet !")
print(f"Prêt pour des défis ML réels et complexes.")

print(f"\n💡 LEÇONS CLÉS :")
print(f"  • Data leakage = ennemi n°1 du ML")
print(f"  • Validation croisée = fiabilité des résultats")
print(f"  • Métriques contextuelles = pertinence métier")
print(f"  • Traçabilité = reproductibilité et confiance")

print(f"\n🎯 PROJET TERMINÉ AVEC SUCCÈS ! 🎉")